[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Reshaping &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Using `wide`, count the hours in which Svalbard was more than 8 degrees colder than Tromso.

In [2]:
wide = readings.pivot(index="time", columns="station", values="celsius")

colder = wide["Tromso"] - wide["Svalbard"] > 8
print(colder.sum())

513


This notebook runs on its own, so the first line builds `wide` again. A comparison with `NaN` is `False`, so Svalbard's silent hours are not counted.

**2.** Make a `pivot_table` of each station's highest reading in each month, with `aggfunc="max"`, and find the month in which every station reached its highest.

In [3]:
by_month = readings.assign(month=readings["time"].dt.month)
highest = by_month.pivot_table(index="station", columns="month", values="celsius", aggfunc="max")

print(highest.idxmax(axis=1).to_dict())
highest

{'Bergen': 7, 'Oslo': 7, 'Svalbard': 7, 'Tromso': 7}


month,1,2,3,4,5,6,7,8,9,10,11,12
station,,,,,,,,,,,,
Bergen,3.0,4.7,8.6,13.3,17.8,20.2,20.8,20.4,18.1,14.3,9.1,5.3
Oslo,1.5,3.2,7.0,11.7,16.2,18.8,19.3,19.0,16.7,12.7,8.0,3.7
Svalbard,-9.5,-7.6,-4.0,0.6,5.1,7.7,8.3,7.9,5.7,1.8,-3.1,-7.0
Tromso,-1.4,0.4,4.2,8.5,13.2,15.6,16.2,15.8,13.6,9.7,4.8,0.9


`idxmax(axis=1)` finds the column with the largest value in each row. Every station's highest reading came in July.

**3.** Make a table of monthly means with `pivot_table`, then `melt` it back to a long table with the columns `station`, `month` and `celsius`, and print its shape.

In [4]:
by_month = readings.assign(month=readings["time"].dt.month)
monthly = by_month.pivot_table(index="station", columns="month", values="celsius").round(1)

monthly_long = monthly.reset_index().melt(id_vars="station", var_name="month", value_name="celsius")
print(monthly_long.shape)
monthly_long.head(3)

(48, 3)


,station,month,celsius
0,Bergen,1,-0.9
1,Oslo,1,-2.4
2,Svalbard,1,-13.4


Four stations by twelve months is 48 rows. `reset_index` makes `station` a column, so that `id_vars` can keep it.

**4.** Make the frost table again without `margins`, and print the station with the most frost hours in March, and how many.

In [5]:
by_month = readings.assign(month=readings["time"].dt.month)
frost = by_month.assign(frost=by_month["celsius"] < 0)
frost_hours = frost.pivot_table(index="station", columns="month", values="frost", aggfunc="sum")

print(frost_hours[3].idxmax(), frost_hours[3].max())

Svalbard 720


`frost_hours[3]` is the column for March, and `idxmax` returns the station with the largest count.

**5.** Make a table with a row for each station and a column for each day of the week, holding the mean reading, with `groupby` on the station and on `readings["time"].dt.day_name()`, which gives each reading's day, and then `unstack`.

In [6]:
day = readings["time"].dt.day_name().rename("day")

readings.groupby(["station", day])["celsius"].mean().unstack().round(2)

day,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
station,,,,,,,
Bergen,8.02,8.02,8.02,8.02,8.02,8.02,7.86
Oslo,6.53,6.52,6.52,6.53,6.52,6.52,6.36
Svalbard,-4.48,-4.48,-4.48,-4.35,-4.48,-4.48,-4.64
Tromso,3.52,3.52,3.52,3.53,3.52,3.52,3.36


`unstack` moves the inner level, the day, into the columns, which come out in alphabetical order because a groupby sorts its keys. Wednesday is lower at every station, because 2025 began and ended on a Wednesday, so it has 53 Wednesdays to the other days' 52, and the extra one is a day in midwinter. Svalbard's Sunday is higher than its other days, because 2 March, its silent day, was a Sunday.

---

&#8592; **Back to:** [Reshaping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/14-reshaping.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
